# Tool Evaluation using the Team Formation Protocol Model
https://www.prismmodelchecker.org/files/atva12mo/

In [1]:
import os
import subprocess
import re
import time

# --- Setup Paths (Ensure these environment variables are correctly set in your environment) ---
mopmc_home = os.getenv('MOPMC_HOME')
storm_home = os.getenv('STORM_HOME')
prism_home = os.getenv('PRISM_HOME')

# Construct full executable paths with checks
prism_executable = os.path.join(prism_home, 'bin', 'prism') if prism_home else None
storm_executable = os.path.join(storm_home, 'build', 'bin', 'storm') if storm_home else None
mopmc_executable = os.path.join(mopmc_home, 'build', 'mopmc') if mopmc_home else None

input_folder = os.path.join(os.getcwd(), 'benchmark_model_team_eval_input')
model_list = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.nm')]
model_list.sort()
model_list

['/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm',
 '/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_4.nm',
 '/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_5.nm']

In [2]:
#create multi_objective properties
C_R_3, P_F_3, P_FA_3, P_FA1_3 = 1.21, 0.60, 0.73, 0.13
C_R_4, P_F_4, P_FA_4, P_FA1_4 = 1.42, 0.71, 0.78, 0.23
C_R_5, P_F_5, P_FA_5, P_FA1_5 = 1.67, 0.83, 0.79, 0.24 
Thresholds = [[C_R_3, P_F_3, P_FA_3, P_FA1_3], [C_R_4, P_F_4, P_FA_4, P_FA1_4], [C_R_5, P_F_5, P_FA_5, P_FA1_5]]                   
for i in range(len(model_list)): 
    C_R, P_F, P_FA, P_FA1 = tuple(Thresholds[i])
    Rw1 = 'R{{"w_1_total"}}>={} [ C ]'.format(C_R)
    Rw2 = 'R{{"w_2_total"}}>={} [ C ]'.format(C_R)
    Pt1 = 'P>={} [ F task1_completed ]'.format(P_F)
    Pt2 = 'P>={} [ F task2_completed ]'.format(P_F)
    Pa1 = 'P>={} [ F agent1_joins_successful_team ]'.format(P_FA)
    Pa2 = 'P>={} [ F agent2_joins_successful_team ]'.format(P_FA)
    Pa3 = 'P>={} [ F agent3_joins_successful_team ]'.format(P_FA)
    Pa11 = 'P>={} [ F agent1_joins_successful_team_of_1 ]'.format(P_FA1)
    Pa12 = 'P>={} [ F agent1_joins_successful_team_of_2 ]'.format(P_FA1)
    Pa13 = 'P>={} [ F agent1_joins_successful_team_of_3 ]'.format(P_FA1)
    property_string_list = [f"multi({Rw1}, {Rw2})",
                            f"multi({Rw1}, {Rw2}, {Pt1}, {Pt2})",
                            f"multi({Rw1}, {Rw2}, {Pt1}, {Pt2}, {Pa1}, {Pa2}, {Pa3})",
                            f"multi({Rw1}, {Rw2}, {Pt1}, {Pt2}, {Pa1}, {Pa2}, {Pa3}, {Pa11}, {Pa12}, {Pa13})"] 
    n_obj_list = [2, 4, 7, 10]
    property_folder = os.path.join(input_folder, 'team_{}_props'.format(i+3))
    os.makedirs(property_folder, exist_ok=True)
    for (ps, n_obj) in zip(property_string_list, n_obj_list):
        property_file_name = property_folder + "/" + f"obj_{n_obj:02d}.pctl"
        with open(property_file_name, 'w') as property_file:
            property_file.write(ps)
        print(f"wrote content to {property_file_name}:")
        #with open(property_file_name, 'r') as property_file:
            # Read all content and print it
            #print(property_file.read().strip()) # .strip() removes trailing whitespace
            #print("---------------------------------------")

wrote content to /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_02.pctl:
wrote content to /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_04.pctl:
wrote content to /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_07.pctl:
wrote content to /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_10.pctl:
wrote content to /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_4_props/obj_02.pctl:
wrote content to /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_4_props/obj_04.pctl:
wrote content to /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_e

In [3]:
"""
model_file_name = model_list[0]
property_file_name = os.path.join(input_folder, 'team_3_props', 'obj_02.pctl')
storm_command  = f"{storm_executable} -e sparse --prism {model_file_name} --prop {property_file_name}"
! $storm_command
"""

'\nmodel_file_name = model_list[0]\nproperty_file_name = os.path.join(input_folder, \'team_3_props\', \'obj_02.pctl\')\nstorm_command  = f"{storm_executable} -e sparse --prism {model_file_name} --prop {property_file_name}"\n! $storm_command\n'

In [4]:
"""
mopmc_achiev = f"{mopmc_executable} -m {model_file_name} -p {property_file_name} -q achievability"
! $mopmc_achiev
"""

'\nmopmc_achiev = f"{mopmc_executable} -m {model_file_name} -p {property_file_name} -q achievability"\n! $mopmc_achiev\n'

In [5]:
"""
%%time
mopmc_convex = f"{mopmc_executable} -m {model_file_name} -p {property_file_name} -q convex"
! $mopmc_convex
"""

'\n%%time\nmopmc_convex = f"{mopmc_executable} -m {model_file_name} -p {property_file_name} -q convex"\n! $mopmc_convex\n'

In [6]:
"""
prism_command = f"{prism_executable} {model_file_name} {property_file_name}"
! $prism_command
"""

'\nprism_command = f"{prism_executable} {model_file_name} {property_file_name}"\n! $prism_command\n'

In [7]:
# --- Define the parsing function ---
def parse_output(output_string, tool_name, query):
    """
    Parses the output of the PRISM tool and extracts relevant metrics.
    """
    extracted_data = {}
    states_match = re.search(r"States:\s*(\d+)", output_string, re.IGNORECASE)
    transitions_match = re.search(r"Transitions:\s*(\d+)", output_string, re.IGNORECASE)
    choices_match = re.search(r"Choices:\s*(\d+)", output_string, re.IGNORECASE)
    loop_count_match = re.search(r"terminates after (\d+) iteration\(s\)", output_string, re.IGNORECASE)
    #loop_count_match = re.search(r"MOPMC main loop count:\s*(\d+)", output_string, re.IGNORECASE)
    loop_count_key = query + '_loop_count_' + tool_name
    
    if states_match:
        extracted_data['States'] = int(states_match.group(1))
    if choices_match:
        extracted_data['Choices'] = int(choices_match.group(1))
    if transitions_match:
        extracted_data['Transitions'] = int(transitions_match.group(1))
    if loop_count_match:
        extracted_data[loop_count_key] = int(loop_count_match.group(1))
    return extracted_data

In [8]:
def run_tool_and_parse(command_list, parser_function=None, tool_name="Tool", query=None):
    """
    Executes a command and, optionally, parses its output using a provided function.

    Args:
        command_list (list): The command and its arguments as a list.
        tool_name (str): A descriptive name for the tool (e.g., "PRISM", "Storm").
        parser_function (callable, optional): A function that takes the tool's
                                               stdout (string) and returns a dict of extracted data.
                                               If None, no parsing is performed.

    Returns:
        dict: A dictionary containing extracted data and a 'success' flag.
              Returns an empty dictionary if the executable is not found or an error occurs.
    """
    extracted_data = {}

    if not command_list or not command_list[0] or not os.path.exists(command_list[0]):
        print(f"Error: {tool_name} executable not found or command list is invalid. "
              f"Attempted executable: {command_list[0] if command_list else 'None'}")
        return extracted_data

    try:
        print(f"Running {tool_name} command: {' '.join(command_list)}")
        result = subprocess.run(
            command_list,
            capture_output=True,
            text=True,
            check=True
        )
        tool_output = result.stdout
        if parser_function:
            parsed_info = parser_function(tool_output, tool_name, query)
            extracted_data.update(parsed_info) # Add parsed info to the result
    except Exception as e:
        print(f"An unexpected error occurred while running {tool_name}: {e}")

    return extracted_data



In [9]:
import subprocess
import os

# Define a constant for the timeout status
TIMEOUT_STATUS = 1
SUCCESS_STATUS = 0
TIMEOUT_SECONDS = 1000

def run_tool_and_parse(command_list, parser_function=None, tool_name="Tool", query=None, timeout_seconds=TIMEOUT_SECONDS):
    """
    Executes a command with a timeout and, optionally, parses its output.

    Args:
        command_list (list): The command and its arguments as a list.
        tool_name (str): A descriptive name for the tool (e.g., "PRISM", "Storm").
        parser_function (callable, optional): A function that takes the tool's
                                              stdout (string) and returns a dict of extracted data.
                                              If None, no parsing is performed.
        query (str, optional): The query or property being checked (for parser context).
        timeout_seconds (int): The maximum time (in seconds) the tool is allowed to run.

    Returns:
        tuple: (extracted_data, return_status).
               extracted_data (dict): Dictionary containing parsed data.
               return_status (int): 0 for success, 1 for timeout, -1 for other errors.
    """
    extracted_data = {}

    # --- Initial Checks ---
    if not command_list or not command_list[0]:
        print(f"Error: Command list is invalid.")
        return extracted_data, -1
    
    # Check if executable exists only if the command is not just a shell alias
    if not os.path.exists(command_list[0]) and '/' in command_list[0] and '.' in command_list[0]:
        print(f"Error: {tool_name} executable not found. Attempted executable: {command_list[0]}")
        return extracted_data, -1

    print(f"Running {tool_name} command with {timeout_seconds}s timeout: {' '.join(command_list)}")

    try:
        # --- Execution with Timeout ---
        result = subprocess.run(
            command_list,
            capture_output=True,
            text=True,
            check=True,  # Raise CalledProcessError for non-zero exit codes
            timeout=timeout_seconds # <<< TIMEOUT ADDED HERE
        )
        
        # If execution is successful and returns 0, proceed with parsing
        tool_output = result.stdout
        if parser_function:
            parsed_info = parser_function(tool_output, tool_name, query)
            extracted_data.update(parsed_info)

        return extracted_data, SUCCESS_STATUS # Success

    except subprocess.TimeoutExpired:
        # --- Handle Timeout ---
        print(f"🚨 Command TIMEOUT: {tool_name} exceeded {timeout_seconds} seconds.")
        # Return extracted_data (empty or partial, depending on context) and status 1
        return extracted_data, TIMEOUT_STATUS

    except subprocess.CalledProcessError as e:
        # --- Handle Non-zero Exit Code ---
        print(f"❌ {tool_name} failed with non-zero exit code {e.returncode}.")
        print(f"STDERR:\n{e.stderr.strip()}")
        # You may optionally parse the stderr here to put error details in extracted_data
        return extracted_data, -1

    except FileNotFoundError:
        # This handles cases where the executable isn't in PATH and os.path.exists failed previously
        print(f"❌ Error: {tool_name} executable '{command_list[0]}' not found in PATH.")
        return extracted_data, -1

    except Exception as e:
        # --- Handle Other Errors ---
        print(f"An unexpected error occurred while running {tool_name}: {e}")
        return extracted_data, -1

In [10]:
def generate_aq_command_list(executable, model, prop_list, tool_name, query='aq'):
    command_list = []
    if query!='aq':
        raise ValueError(f"Unsupported query: {query}!")
    for prop in prop_list:
        if tool_name == 'mopmc':
            command = [executable, '-m', model, '-p', prop, '-q', 'achievability']            
        elif tool_name == "mopmc_cpu":
            command = [executable, '-m', model, '-p', prop, '-q', 'achievability', '-v', 'standard']
        elif tool_name == "storm":
            command = [executable, '--prism', model, '--prop', prop]
        elif tool_name == "prism":        
            command = [executable, model, prop]
        else:
             raise ValueError(f"Incorrect tool name for achievability query: {tool_name}!")
        command_list.append(command)
    return command_list


def generate_cq_command_list(executable, model, prop_list, tool_name, query='ccq'):
    command_list = []
    
    for prop in prop_list:
        if query=='ccq':
            if tool_name == 'mopmc':
                command = [executable, '-m', model, '-p', prop, '-q', 'convex']            
            elif tool_name == "mopmc_cpu":
                command = [executable, '-m', model, '-p', prop, '-q', 'convex', '-v', 'standard']
            else:
                 raise ValueError(f"Incorrect tool name for convex query: {tool_name}!")
        elif query=='ucq':    
            if tool_name == 'mopmc':
                command = [executable, '-m', model, '-p', prop, '-q', 'convex', '-c', 'n']            
            elif tool_name == "mopmc_cpu":
                command = [executable, '-m', model, '-p', prop, '-q', 'convex', '-c', 'n', '-v', 'standard']
            else:
                 raise ValueError(f"Incorrect tool name for convex query: {tool_name}!")
        else:
            raise ValueError(f"Unsupported query: {query}!")
        command_list.append(command)
    return command_list

In [11]:
import os

def generate_prop_nested_list(model_list):
    """
    Constructs a nested list containing the full paths to all .pctl files 
    found in the property folder, sorted by filename (e.g., obj_02, obj_04, etc.).
    """
    nested_list = []
    
    for model_path in model_list:
        # 1. Get components and construct the property folder path
        folder_path = os.path.dirname(model_path)
        base_name = os.path.basename(model_path)
        team_string = base_name.split('.')[0] 
        prop_folder_path = os.path.join(folder_path, team_string + "_props")
        
        current_model_file_paths = [] 
        
        # Check if the folder exists
        if os.path.isdir(prop_folder_path):
            # 2. List all contents of the folder
            all_entries = os.listdir(prop_folder_path)
            
            # 3. Filter for .pctl files and create full paths (List Comprehension)
            pctl_full_paths = [
                os.path.join(prop_folder_path, entry) 
                for entry in all_entries 
                if entry.endswith('.pctl') and os.path.isfile(os.path.join(prop_folder_path, entry))
            ]
            
            # 4. SORT THE LIST ALPHABETICALLY (Numerically due to padding)
            pctl_full_paths.sort() 
            
            current_model_file_paths = pctl_full_paths
            #print(f"Found {len(pctl_full_paths)} sorted .pctl files in {prop_folder_path}")
            #print(f"   First file: {pctl_full_paths[0]}")
            
        else:
            print(f"⚠️ Warning: Folder not found: {prop_folder_path}")
        
        # 5. Append the sorted list of full paths
        nested_list.append(current_model_file_paths)
            
    return nested_list
    
nested_prop_list = generate_prop_nested_list(model_list)
nested_prop_list    

[['/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_02.pctl',
  '/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_04.pctl',
  '/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_07.pctl',
  '/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_10.pctl'],
 ['/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_4_props/obj_02.pctl',
  '/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_4_props/obj_04.pctl',
  '/home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_4_props/obj_07.pctl',
  '/home/guoxin/PycharmProjects/realtime-drl-switch/sc

In [12]:
model_name_rex = r'team_(\d+)\.nm'
model_label = 'n_sensors'
obj_num = 'n_objs'
def run_experiment(executable, model_list, prop_list_all_models, tool_name, query='aq'):

    def extract_model_id(mode_path):
        match = re.search(model_name_rex, os.path.basename(mode_path))
        model_id = None
        if match:
            model_id = int(match.group(1)) 
        else:
            print(f"Warning: Could not extract integer ID from model: {mode_path}")    
        return model_id
    
    def extract_n_objs(command):
        pattern = re.compile(r'(\d{2})\.pctl')
        extracted_number = None
        for elem in command:
            match = pattern.search(elem)     
            if match:
                # Found the string containing the property file name
                extracted_number = match.group(1) 
                break
        if extracted_number is not None:
            integer_value = int(extracted_number)
        return integer_value
    
    if (len(model_list) != len(prop_list_all_models)):
        raise ValueError("num of model files != num of property files")
    
    results = []
    for (model, prop_list) in zip(model_list, prop_list_all_models):
        if query=='aq':
            command_list = generate_aq_command_list(executable, model, prop_list, tool_name, query)
        else:
            command_list = generate_cq_command_list(executable, model, prop_list, tool_name, query)
        runtime_key = query + '_run_time_' + tool_name   
        
        for command in command_list:            
                
            start_time = time.perf_counter()
            data, STATUS = run_tool_and_parse(command, parse_output, tool_name, query)
            end_time = time.perf_counter()
            data[model_label] = extract_model_id(model)
            data[obj_num] = extract_n_objs(command)       
            if data[model_label] is None or data[obj_num] is None:
                raise ValueError("model label or obj num is none")
            if STATUS == SUCCESS_STATUS:
                data["timeout"] = "no"
            elif STATUS == TIMEOUT_STATUS:
                data["timeout"] = "yes"
                results.append(data)
                break # Skip the rest of the inner loop (commands)
            else: 
                raise ValueError("Error occurs in tool runing!")
            
            data[runtime_key] = end_time - start_time      
            results.append(data)
            
    return results

In [13]:
mopmc_gpu_aq_results= run_experiment(mopmc_executable, model_list, nested_prop_list, "mopmc", "aq")
mopmc_gpu_aq_results.sort(key= lambda item: (item[model_label], item[obj_num]))
print(f"\nMOPMC-GPU Achievability Query Results:")
print(mopmc_gpu_aq_results)

Running mopmc command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_02.pctl -q achievability
Running mopmc command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_04.pctl -q achievability
Running mopmc command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool

In [14]:
mopmc_cpu_aq_results= run_experiment(mopmc_executable, model_list, nested_prop_list, "mopmc_cpu", "aq")
mopmc_cpu_aq_results.sort(key= lambda item: (item[model_label], item[obj_num]))
print(f"\nMOPMC-CPU Achievability Query Results:")
print(mopmc_cpu_aq_results)

Running mopmc_cpu command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_02.pctl -q achievability -v standard
Running mopmc_cpu command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_04.pctl -q achievability -v standard
Running mopmc_cpu command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProje

In [15]:
mopmc_gpu_ccq_results= run_experiment(mopmc_executable, model_list, nested_prop_list, "mopmc", "ccq")
mopmc_gpu_ccq_results.sort(key= lambda item: (item[model_label], item[obj_num]))
print(f"\nMOPMC-GPU Convex Query Results:")
print(mopmc_gpu_ccq_results)

Running mopmc command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_02.pctl -q convex
Running mopmc command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_04.pctl -q convex
Running mopmc command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/be

In [16]:
mopmc_cpu_ccq_results= run_experiment(mopmc_executable, model_list, nested_prop_list, "mopmc_cpu", "ccq")
mopmc_cpu_ccq_results.sort(key= lambda item: (item[model_label], item[obj_num]))
print(f"\nMOPMC-CPU Convex Query Results:")
print(mopmc_cpu_ccq_results)

Running mopmc_cpu command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_02.pctl -q convex -v standard
Running mopmc_cpu command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_04.pctl -q convex -v standard
Running mopmc_cpu command with 1000s timeout: /home/guoxin/CLionProjects/mopmc-dev-v3/build/mopmc -m /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm -p /home/guoxin/PycharmProjects/realtime-d

In [17]:
#mopmc_gpu_ucq_results= run_experiment(mopmc_executable, model_list, nested_prop_list, "mopmc", "ucq")
#mopmc_gpu_ucq_results.sort(key= lambda item: (item[model_label], item[obj_num]))
#print(f"\nMOPMC-GPU (Unconstrained) Convex Query Results:")
#print(mopmc_gpu_ucq_results)

In [18]:
#mopmc_cpu_ucq_results= run_experiment(mopmc_executable, model_list, nested_prop_list, "mopmc_cpu", "ucq")
#mopmc_cpu_ucq_results.sort(key= lambda item: (item[model_label], item[obj_num]))
#print(f"\nMOPMC-CPU (Unconstrained) Convex Query Results:")
#print(mopmc_cpu_ucq_results)

In [19]:
storm_results= run_experiment(storm_executable, model_list, nested_prop_list, "storm")
storm_results.sort(key= lambda item: (item[model_label], item[obj_num]))
print(f"\nStorm Results:")
print(storm_results)

Running storm command with 1000s timeout: /home/guoxin/Downloads/storm/build/bin/storm --prism /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm --prop /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_02.pctl
Running storm command with 1000s timeout: /home/guoxin/Downloads/storm/build/bin/storm --prism /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm --prop /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_04.pctl
Running storm command with 1000s timeout: /home/guoxin/Downloads/storm/build/bin/storm --prism /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm --prop /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_

In [20]:
prism_results = run_experiment(prism_executable, model_list, nested_prop_list, "prism")
prism_results.sort(key= lambda item: item[model_label])
print(f"\nPRISM Results:")
print(prism_results)

Running prism command with 1000s timeout: /home/guoxin/Downloads/prism-4.4-linux64/bin/prism /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_02.pctl
Running prism command with 1000s timeout: /home/guoxin/Downloads/prism-4.4-linux64/bin/prism /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_props/obj_04.pctl
Running prism command with 1000s timeout: /home/guoxin/Downloads/prism-4.4-linux64/bin/prism /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3.nm /home/guoxin/PycharmProjects/realtime-drl-switch/scripts/tool_evaluation/benchmark_model_team_eval_input/team_3_prop

In [21]:
from itertools import zip_longest

# Assuming the result lists are defined in your current scope:
# storm_results
# prism_results
# mopmc_gpu_aq_results
# mopmc_cpu_aq_results   # Assumed to be available
# mopmc_gpu_ccq_results
# mopmc_cpu_ccq_results  # Assumed to be available

def merge_results_safely(storm_r, prism_r, mopmc_g_aq_r, mopmc_c_aq_r, mopmc_g_ccq_r, mopmc_c_ccq_r):
    """
    Merges dictionaries from six lists of potentially unequal length into a single list
    of combined dictionaries. Uses zip_longest with an empty dictionary fillvalue.
    """
    combined_results = []
    FILL_VALUE = {} 

    print("Starting merge using zip_longest...")
    
    # Use zip_longest to iterate through all lists until the longest one is done
    for d0, d1, d2, d3, d4, d5 in zip_longest(
        storm_r, 
        prism_r, 
        mopmc_g_aq_r, 
        mopmc_c_aq_r,
        mopmc_g_ccq_r, 
        mopmc_c_ccq_r,
        fillvalue=FILL_VALUE
    ):
        # Initialise combined_dict. If the first item (d0) is the fill value, start with {}.
        combined_dict = d0.copy() if d0 is not FILL_VALUE else {}
        
        # Safely update with all other dictionaries.
        # Calling update({}) is harmless if d1 through d5 are the FILL_VALUE.
        combined_dict.update(d1)
        combined_dict.update(d2)
        combined_dict.update(d3)
        combined_dict.update(d4)
        combined_dict.update(d5)
        
        # Append only if the resulting dictionary has content.
        if combined_dict:
            combined_results.append(combined_dict)

    return combined_results

# Execute the merge function
combined_results = merge_results_safely(
    storm_results, 
    prism_results, 
    mopmc_gpu_aq_results, 
    mopmc_cpu_aq_results,  # Assuming this list exists
    mopmc_gpu_ccq_results, 
    mopmc_cpu_ccq_results  # Assuming this list exists
)

print(f"\n✅ Merge complete. Total combined rows: {len(combined_results)}")

Starting merge using zip_longest...

✅ Merge complete. Total combined rows: 12


In [22]:
import pandas as pd
combined_results_df = pd.DataFrame(combined_results)
# Calculate GPU runtime per iteration
combined_results_df.loc[:, 'ccq_run_time_mopmc_per_it'] = (
    combined_results_df['ccq_run_time_mopmc'] / 
    combined_results_df['ccq_loop_count_mopmc']
)

# Calculate CPU runtime per iteration
combined_results_df.loc[:, 'ccq_run_time_mopmc_cpu_per_it'] = (
    combined_results_df['ccq_run_time_mopmc_cpu'] / 
    combined_results_df['ccq_loop_count_mopmc_cpu']
)

# --- Remaining Code (Looks correct) ---

# Reorder columns (This creates a new DataFrame, so subsequent ops are fine)
new_column_order = [model_label, obj_num, 'States', 'Choices', 'Transitions', 
                    'aq_run_time_storm', 'aq_run_time_prism', 
                    'aq_run_time_mopmc', 'aq_run_time_mopmc_cpu', 
                    'ccq_run_time_mopmc', 'ccq_run_time_mopmc_cpu',
                    'ccq_loop_count_mopmc' ,#'ccq_run_time_mopmc_per_it', 
                    'ccq_loop_count_mopmc_cpu' #'ccq_run_time_mopmc_cpu_per_it',
                   ]
combined_results_df = combined_results_df[new_column_order]

# Round numerical columns
num_columns = ['aq_run_time_storm', 'aq_run_time_prism', 
               'aq_run_time_mopmc', 'aq_run_time_mopmc_cpu', 
               'ccq_run_time_mopmc', #'ccq_run_time_mopmc_per_it',
               'ccq_run_time_mopmc_cpu' #, 'ccq_run_time_mopmc_cpu_per_it'
              ]
combined_results_df[num_columns] = combined_results_df[num_columns].round(2)

combined_results_df

,n_sensors,n_objs,States,Choices,Transitions,aq_run_time_storm,aq_run_time_prism,aq_run_time_mopmc,aq_run_time_mopmc_cpu,ccq_run_time_mopmc,ccq_run_time_mopmc_cpu,ccq_loop_count_mopmc,ccq_loop_count_mopmc_cpu
0,3,2,12475,14935,15228,0.17,2.20,0.42,0.14,0.40,0.19,16,16
1,3,4,12475,14935,15228,0.09,4.39,0.36,0.15,0.37,0.11,19,19
2,3,7,12475,14935,15228,0.13,4.95,0.30,0.17,0.33,0.13,17,17
3,3,10,12475,14935,15228,0.24,5.52,0.40,0.16,0.70,0.62,200,200
4,4,2,96665,115289,116464,0.42,62.07,0.63,0.46,0.70,0.52,12,12
5,4,4,96665,115289,116464,0.59,131.29,0.74,0.48,0.73,1.24,16,81
6,4,7,96665,115289,116464,8.06,174.34,0.90,0.68,1.06,2.33,81,122
7,4,10,96665,115289,116464,NaN,190.68,1.33,1.42,1.44,2.19,33,57
8,5,2,907993,1078873,1084752,5.56,NaN,5.48,5.17,5.59,6.40,14,14
9,5,4,907993,1078873,1084752,6.87,NaN,5.95,5.98,6.04,7.38,15,14


In [23]:
print(combined_results_df)

    n_sensors  n_objs  States  Choices  Transitions  aq_run_time_storm  \
0           3       2   12475    14935        15228               0.17   
1           3       4   12475    14935        15228               0.09   
2           3       7   12475    14935        15228               0.13   
3           3      10   12475    14935        15228               0.24   
4           4       2   96665   115289       116464               0.42   
5           4       4   96665   115289       116464               0.59   
6           4       7   96665   115289       116464               8.06   
7           4      10   96665   115289       116464                NaN   
8           5       2  907993  1078873      1084752               5.56   
9           5       4  907993  1078873      1084752               6.87   
10          5       7  907993  1078873      1084752              23.01   
11          5      10  907993  1078873      1084752                NaN   

    aq_run_time_prism  aq_run_time_mo